# Diabetes Risk Prediction
Pipeline for generating diabetes dataset and training machine learning models.

### Model Specifications & Pipeline Overview
- **Target Variable**: `diabetes` (0: Low Risk, 1: High Risk)
- **Export Dataset**: `models/diabetes_dataset.csv`
- **Algorithms Trained**: Logistic Regression, Decision Tree, Random Forest, XGBoost, Support Vector Machine (SVM)
- **Artifacts Generated**: `diabetes_*.pkl` model files, `scaler.pkl`, `feature_names.json`, `model_metrics.json`



In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Resolve models directory path safely regardless of CWD
if os.path.basename(os.getcwd()) == 'notebooks':
    models_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))
else:
    models_dir = os.path.abspath(os.path.join(os.getcwd(), 'models'))

os.makedirs(models_dir, exist_ok=True)
print(f"Target Models Directory: {models_dir}")



In [ ]:
# Generate Diabetes Dataset with Specific Clinical Features
np.random.seed(42)
num_records = 2500

ages = np.random.randint(18, 85, size=num_records)
genders = np.random.choice(['male', 'female', 'other'], size=num_records, p=[0.48, 0.48, 0.04])
heights = np.random.normal(170, 10, size=num_records)
weights = np.clip(heights * 0.45 + np.random.normal(0, 10, size=num_records), 40, 150)
bmis = weights / ((heights / 100) ** 2)

smokers = np.random.choice(['yes', 'no'], size=num_records, p=[0.20, 0.80])
alcohol_use = np.random.choice(['low', 'moderate', 'high'], size=num_records, p=[0.60, 0.30, 0.10])
activities = np.random.choice(['sedentary', 'moderate', 'active'], size=num_records, p=[0.35, 0.45, 0.20])
sleep_hours = np.clip(np.random.normal(7, 1.2, size=num_records), 4, 10)

systolic = np.clip(110 + (bmis - 22) * 1.2 + (ages - 30) * 0.3 + np.random.normal(0, 8, size=num_records), 85, 200).astype(int)
diastolic = np.clip(70 + (bmis - 22) * 0.8 + (ages - 30) * 0.15 + np.random.normal(0, 6, size=num_records), 55, 120).astype(int)
cholesterol = np.clip(160 + (bmis - 22) * 2.0 + (ages - 30) * 0.8 + np.random.normal(0, 15, size=num_records), 100, 380).astype(int)

# Glucose spikes & Insulin correlation
glucose = 85 + (bmis - 22) * 1.5 + (ages - 30) * 0.4 + np.random.normal(0, 12, size=num_records)
diabetes_spikes = (bmis > 28) & (ages > 45) & (np.random.rand(num_records) < 0.45)
glucose[diabetes_spikes] += np.random.randint(45, 125, size=np.sum(diabetes_spikes))
glucose = np.clip(glucose, 55, 280).astype(int)

insulin = np.clip(5 + (glucose - 85) * 0.18 + (bmis - 22) * 0.6 + np.random.normal(0, 3, size=num_records), 2, 55).astype(int)
heart_rate = np.clip(65 + (bmis - 22) * 0.4 + np.random.normal(0, 8, size=num_records), 45, 130).astype(int)

# Target label based on clinical thresholds
y_diabetes = ((glucose > 130) | ((bmis > 30) & (glucose > 115))).astype(int)

df_diabetes = pd.DataFrame({
    'age': ages, 'gender': genders, 'height': heights, 'weight': weights, 'bmi': bmis,
    'smoking': smokers, 'alcohol': alcohol_use, 'physical_activity': activities,
    'sleep_duration': sleep_hours, 'bp_systolic': systolic, 'bp_diastolic': diastolic,
    'cholesterol': cholesterol, 'glucose': glucose, 'insulin': insulin, 'heart_rate': heart_rate,
    'diabetes': y_diabetes
})

csv_path = os.path.join(models_dir, "diabetes_dataset.csv")
df_diabetes.to_csv(csv_path, index=False)
print(f"Saved Diabetes Dataset to: {csv_path} ({len(df_diabetes)} records, {y_diabetes.sum()} positive cases)")



In [ ]:
# Preprocessing & Feature Alignment
categorical_cols = ['gender', 'smoking', 'alcohol', 'physical_activity']
numerical_cols = ['age', 'height', 'weight', 'bmi', 'sleep_duration', 'bp_systolic', 'bp_diastolic', 'cholesterol', 'glucose', 'insulin', 'heart_rate']

cat_categories = {
    'gender': ['male', 'female', 'other'],
    'smoking': ['yes', 'no'],
    'alcohol': ['low', 'moderate', 'high'],
    'physical_activity': ['sedentary', 'moderate', 'active']
}

def preprocess_features(df_input):
    X_num = df_input[numerical_cols].values
    encoded_cats = []
    for col, cats in cat_categories.items():
        for category in cats:
            encoded_cats.append((df_input[col] == category).astype(float).values)
    X_cat = np.column_stack(encoded_cats)
    return np.column_stack([X_num, X_cat])

X = preprocess_features(df_diabetes)
y = df_diabetes['diabetes'].values

# Fit and export global StandardScaler
scaler = StandardScaler()
scaler.fit(df_diabetes[numerical_cols].values)

scaler_path = os.path.join(models_dir, "scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"StandardScaler updated at: {scaler_path}")

# Export Feature Names
feature_names = numerical_cols.copy()
for col, cats in cat_categories.items():
    for category in cats:
        feature_names.append(f"{col}_{category}")

feature_names_path = os.path.join(models_dir, "feature_names.json")
with open(feature_names_path, "w") as f:
    json.dump(feature_names, f, indent=2)
print(f"Feature names saved at: {feature_names_path}")



In [ ]:
# Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical features (first 11 columns)
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[:, :11] = scaler.transform(X_train[:, :11])
X_test_scaled[:, :11] = scaler.transform(X_test[:, :11])

algorithms = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'decision_tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'xgboost': XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    'svm': SVC(probability=True, C=1.0, kernel='rbf', random_state=42)
}

metrics_file = os.path.join(models_dir, "model_metrics.json")
metrics_report = {}
if os.path.exists(metrics_file):
    try:
        with open(metrics_file, "r") as f:
            metrics_report = json.load(f)
    except Exception:
        metrics_report = {}

metrics_report['diabetes'] = {}

print(f"\nTraining models for target domain: diabetes")
for alg_name, clf in algorithms.items():
    clf.fit(X_train_scaled, y_train)
    
    model_filename = os.path.join(models_dir, f"diabetes_{alg_name}.pkl")
    with open(model_filename, "wb") as f:
        pickle.dump(clf, f)
        
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    metrics_report['diabetes'][alg_name] = {
        'accuracy': round(float(acc), 4),
        'precision': round(float(prec), 4),
        'recall': round(float(rec), 4),
        'f1_score': round(float(f1), 4),
        'roc_auc': round(float(auc), 4)
    }
    print(f"  [{alg_name}] Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

with open(metrics_file, "w") as f:
    json.dump(metrics_report, f, indent=2)

print(f"\nCompleted training for diabetes. Updated metrics written to: {metrics_file}")

